# ResikIn Waste Classifier - Model Fine-Tuning

Notebook ini digunakan untuk melatih (fine-tune) model CLIP agar lebih pintar mendeteksi sampah. Kita akan:
1. Mengunduh dataset dari Roboflow langsung ke server Google Colab.
2. Menjalankan skrip persiapan data.
3. Menjalankan proses training.

## 1. Persiapan Lingkungan (Install Dependencies)

In [ ]:
!pip install -q roboflow transformers torch torchvision scikit-learn pillow matplotlib tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 112.5 MB/s eta 0:00:00


## 2. Unduh Dataset dari Roboflow

Sesuai dengan API yang Anda berikan, ini akan mengunduh dataset ke dalam folder `dataset_mentah` di Colab.

In [ ]:
import os
from roboflow import Roboflow

# Buat folder untuk dataset mentah
os.makedirs("dataset_mentah", exist_ok=True)
os.chdir("dataset_mentah")

# Download dari Roboflow menggunakan key Anda
rf = Roboflow(api_key="CsZFCzVXJL8qqYDhx0hR")
project = rf.workspace("project-ia-andzk").project("classification-image-6zihm")
version = project.version(2)
dataset = version.download("folder")

print(f"\nDataset berhasil diunduh di path: {dataset.location}")
os.chdir("..")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to classification-image-2 in folder:: 100%|██████████| 298/298 [00:00<00:00, 11043.20it/s]


Dataset berhasil diunduh di path: /content/dataset_mentah/classification-image-2


## 3. Clone Repository Anda

Kita perlu mengunduh skrip `train.py` dan `prepare_dataset.py` yang sudah dibuat. Pastikan Anda sudah me-*push* kode ke repositori GitHub Anda.

In [ ]:
# Ganti URL ini dengan URL repository GitHub baru Anda yang berisi resikin-waste-classifier
# Jika repo private, hapus baris ini dan unggah file secara manual.
!git clone https://github.com/Hanafi-Sh/resikin-ai.git repo_ai

# Jika Anda belum push repo baru, Anda bisa mengunggah folder resikin-waste-classifier
# secara manual ke Colab dan mengubah path di sel-sel berikutnya.

Cloning into 'repo_ai'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 31 (delta 5), reused 30 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 17.68 KiB | 17.68 MiB/s, done.
Resolving deltas: 100% (5/5), done.


## 4. Rapikan Dataset

Roboflow mengunduh dataset dengan nama folder tertentu. Kita akan menggunakan skrip `prepare_dataset.py` untuk memindahkannya ke dalam struktur yang diharapkan oleh skrip training.

In [ ]:
import glob

# Mencari folder hasil download (nama foldernya biasanya sama dengan nama project)
download_dirs = glob.glob("dataset_mentah/*/")
roboflow_dir = download_dirs[0] if download_dirs else ""

if roboflow_dir:
    print(f"Menggunakan direktori Roboflow: {roboflow_dir}")
    !python repo_ai/src/preprocessing/prepare_dataset.py --roboflow_dir "{roboflow_dir}" --output_dir "./data"
else:
    print("Folder dataset tidak ditemukan!")

Menggunakan direktori Roboflow: dataset_mentah/classification-image-2/
🔄 Organizing Roboflow dataset...
⚠️  Split 'valid' atau 'test' tidak ditemukan di Roboflow. Akan melakukan auto-split 80/10/10 dari data train.
  [train] vide: 135 images
  [val] vide: 16 images
  [test] vide: 16 images
  [train] pleine: 102 images
  [val] pleine: 12 images
  [test] pleine: 12 images

✅ Dataset siap digunakan untuk training!

📊 Dataset Statistics:
----------------------------------------
  train  / pleine          :   102 images
  train  / vide            :   135 images
  val    / pleine          :    12 images
  val    / vide            :    16 images
  test   / pleine          :    12 images
  test   / vide            :    16 images


## 5. Mulai Training! 🚀

Ini akan melatih model CLIP. Waktu yang dibutuhkan sekitar 20-40 menit tergantung jumlah dataset dan GPU (pastikan Runtime -> Change runtime type -> Hardware accelerator: **T4 GPU**).

In [ ]:
!python repo_ai/scripts/train.py --data_dir "./data" --output_dir "./models" --epochs 10 --batch_size 32

Device: cuda
Loading CLIP ViT-B/32...
config.json: 4.19kB [00:00, 12.4MB/s]
pytorch_model.bin: 100% 605M/605M [00:04<00:00, 136MB/s]
Loading weights: 100% 398/398 [00:00<00:00, 808.87it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
model.safetensors:   8% 50.0M/605M [00:01<00:06, 88.4MB/s]
preprocessor_config.json: 100% 316/316 [00:00<00:00, 966kB/s]
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue usi

## 6. Unduh Hasil Model

Setelah training selesai, kita akan mengompres folder `models/` agar Anda bisa mengunduhnya ke laptop Anda.

In [ ]:
import shutil
from google.colab import files

# Zip folder models
shutil.make_archive("hasil_model_clip", 'zip', "./models")

# Download zip file
files.download("hasil_model_clip.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>